In [1]:
from pathlib import Path

NVMS_ROOT = Path("/home/jovyan/work-easi-eds/nvms_runs")
SCRATCH_ROOT = Path("/home/jovyan/scratch/eds/tiles")

OUT_CSV = Path("/home/jovyan/work-easi-eds/nvms_runs/run_availability_index.csv")

# Optional: count only clr products (recommended if you always run --sr-only-clr/--fc-only-clr)
ONLY_CLR = True


In [2]:
import re
import pandas as pd

PAT = re.compile(
    r"^(?P<prefix>[a-z0-9]+)_(?P<scene>p\d{3}r\d{3})_d(?P<start>\d{8})(?P<end>\d{8})_(?P<tag>[a-z0-9]+)$",
    re.IGNORECASE
)

def find_runnable_jobs(nvms_root: Path) -> pd.DataFrame:
    rows = []
    for run_dir in sorted(nvms_root.glob("run*")):
        if not run_dir.is_dir():
            continue
        for job_dir in sorted(run_dir.iterdir()):
            if not job_dir.is_dir():
                continue

            m = PAT.match(job_dir.name)
            if not m:
                continue

            rows.append({
                "run": run_dir.name,
                "job_dir": str(job_dir),
                "job_name": job_dir.name,
                "scene": m.group("scene").lower(),          # p089r078
                "start_yyyymmdd": m.group("start"),
                "end_yyyymmdd": m.group("end"),
                "start_iso": f"{m.group('start')[:4]}-{m.group('start')[4:6]}-{m.group('start')[6:]}",
                "end_iso": f"{m.group('end')[:4]}-{m.group('end')[4:6]}-{m.group('end')[6:]}",
                "tag": m.group("tag").lower(),
            })

    df = pd.DataFrame(rows).sort_values(["run", "scene", "start_yyyymmdd", "end_yyyymmdd"]).reset_index(drop=True)
    return df

jobs_df = find_runnable_jobs(NVMS_ROOT)
print("Detected runnable folders:", len(jobs_df))
jobs_df.head(10)


Detected runnable folders: 219


,run,job_dir,job_name,scene,start_yyyymmdd,end_yyyymmdd,start_iso,end_iso,tag
0,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r078_d2023030620231024_dlwm6,p089r078,20230306,20231024,2023-03-06,2023-10-24,dlwm6
1,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r084_d2023012520231024_dlwm6,p089r084,20230125,20231024,2023-01-25,2023-10-24,dlwm6
2,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r088_d2023010820231015_dlwm5,p090r088,20230108,20231015,2023-01-08,2023-10-15,dlwm5
3,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r090_d2023010820230727_dlwm5,p090r090,20230108,20230727,2023-01-08,2023-07-27,dlwm5
4,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r076_d2023030420231022_dlwm6,p091r076,20230304,20231022,2023-03-04,2023-10-22,dlwm6
5,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r087_d2023030420230928_dlwm5,p091r087,20230304,20230928,2023-03-04,2023-09-28,dlwm5
6,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r089_d2023011520230216_dlwm5,p091r089,20230115,20230216,2023-01-15,2023-02-16,dlwm5
7,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r090_d2023011520230904_dlwm5,p091r090,20230115,20230904,2023-01-15,2023-09-04,dlwm5
8,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r075_d2023041220231029_dlwm6,p092r075,20230412,20231029,2023-04-12,2023-10-29,dlwm6
9,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r084_d2023022420230928_dlwm5,p092r084,20230224,20230928,2023-02-24,2023-09-28,dlwm5


In [3]:
from datetime import datetime
from collections import defaultdict

DATE_RE = re.compile(r"_(\d{8})_")

def parse_date_from_filename(name: str):
    m = DATE_RE.search(name)
    if not m:
        return None
    return datetime.strptime(m.group(1), "%Y%m%d").date()

def scan_product(scene: str, product: str, only_clr: bool = True):
    """
    product: "sr" or "fc"
    Returns: count, min_date, max_date
    """
    root = SCRATCH_ROOT / scene / product
    if not root.exists():
        return 0, None, None

    dates = []
    for p in root.rglob("*.tif"):
        n = p.name.lower()
        if only_clr and "_clr" not in n:
            continue
        d = parse_date_from_filename(p.name)
        if d:
            dates.append(d)

    if not dates:
        return 0, None, None
    return len(dates), min(dates), max(dates)

def scan_fmask(scene: str):
    """
    Best-effort: look for a mask folder or fmask in filename.
    You can tighten this once we confirm how your fmask is stored.
    """
    scene_root = SCRATCH_ROOT / scene
    if not scene_root.exists():
        return 0, None, None

    hits = []
    for p in scene_root.rglob("*.tif"):
        n = p.name.lower()
        if "fmask" in n or "mask" in n:
            d = parse_date_from_filename(p.name)
            if d:
                hits.append(d)

    if not hits:
        return 0, None, None
    return len(hits), min(hits), max(hits)

# Build a tile availability index for all tiles found in jobs
unique_scenes = sorted(jobs_df["scene"].unique())
rows = []
for scene in unique_scenes:
    sr_n, sr_min, sr_max = scan_product(scene, "sr", only_clr=ONLY_CLR)
    fc_n, fc_min, fc_max = scan_product(scene, "fc", only_clr=ONLY_CLR)
    fm_n, fm_min, fm_max = scan_fmask(scene)

    rows.append({
        "scene": scene,
        "sr_tiles": sr_n,
        "sr_min": sr_min,
        "sr_max": sr_max,
        "fc_tiles": fc_n,
        "fc_min": fc_min,
        "fc_max": fc_max,
        "fmask_tiles": fm_n,
        "fmask_min": fm_min,
        "fmask_max": fm_max,
        "has_fmask": fm_n > 0,
    })

avail_df = pd.DataFrame(rows).sort_values("scene").reset_index(drop=True)
avail_df.head(25)


,scene,sr_tiles,sr_min,sr_max,fc_tiles,fc_min,fc_max,fmask_tiles,fmask_min,fmask_max,has_fmask
0,p089r078,0,None,None,0,None,None,0,None,None,False
1,p089r084,0,None,None,0,None,None,0,None,None,False
2,p090r088,0,None,None,0,None,None,0,None,None,False
3,p090r090,0,None,None,0,None,None,0,None,None,False
4,p091r076,0,None,None,0,None,None,0,None,None,False
5,p091r087,0,None,None,0,None,None,0,None,None,False
6,p091r089,0,None,None,0,None,None,0,None,None,False
7,p091r090,0,None,None,0,None,None,0,None,None,False
8,p092r075,0,None,None,0,None,None,1,2016-02-20,2016-02-20,True
9,p092r084,0,None,None,0,None,None,0,None,None,False


In [4]:
master_df = jobs_df.merge(avail_df, on="scene", how="left")

# Handy “quick status” — SR only
master_df["has_sr_100plus"] = master_df["sr_tiles"].fillna(0) >= 80
master_df["eligible_sr_100plus"] = master_df["has_sr_100plus"]

master_df.head(10)


,run,job_dir,job_name,scene,start_yyyymmdd,end_yyyymmdd,start_iso,end_iso,tag,sr_tiles,...,sr_max,fc_tiles,fc_min,fc_max,fmask_tiles,fmask_min,fmask_max,has_fmask,has_sr_100plus,eligible_sr_100plus
0,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r078_d2023030620231024_dlwm6,p089r078,20230306,20231024,2023-03-06,2023-10-24,dlwm6,0,...,None,0,None,None,0,None,None,False,False,False
1,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p089r084_d2023012520231024_dlwm6,p089r084,20230125,20231024,2023-01-25,2023-10-24,dlwm6,0,...,None,0,None,None,0,None,None,False,False,False
2,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r088_d2023010820231015_dlwm5,p090r088,20230108,20231015,2023-01-08,2023-10-15,dlwm5,0,...,None,0,None,None,0,None,None,False,False,False
3,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p090r090_d2023010820230727_dlwm5,p090r090,20230108,20230727,2023-01-08,2023-07-27,dlwm5,0,...,None,0,None,None,0,None,None,False,False,False
4,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r076_d2023030420231022_dlwm6,p091r076,20230304,20231022,2023-03-04,2023-10-22,dlwm6,0,...,None,0,None,None,0,None,None,False,False,False
5,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r087_d2023030420230928_dlwm5,p091r087,20230304,20230928,2023-03-04,2023-09-28,dlwm5,0,...,None,0,None,None,0,None,None,False,False,False
6,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r089_d2023011520230216_dlwm5,p091r089,20230115,20230216,2023-01-15,2023-02-16,dlwm5,0,...,None,0,None,None,0,None,None,False,False,False
7,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p091r090_d2023011520230904_dlwm5,p091r090,20230115,20230904,2023-01-15,2023-09-04,dlwm5,0,...,None,0,None,None,0,None,None,False,False,False
8,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r075_d2023041220231029_dlwm6,p092r075,20230412,20231029,2023-04-12,2023-10-29,dlwm6,0,...,None,0,None,None,1,2016-02-20,2016-02-20,True,False,False
9,run1,/home/jovyan/work-easi-eds/nvms_runs/run1/lzol...,lzolre_p092r084_d2023022420230928_dlwm5,p092r084,20230224,20230928,2023-02-24,2023-09-28,dlwm5,0,...,None,0,None,None,0,None,None,False,False,False


In [5]:
from datetime import datetime

PIPELINE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py"
TILE_SHP = "/home/jovyan/assets/eds_lsat_grid_min_max.shp"
SPAN_YEARS = 10

# threshold for "enough" SR tiles
SR_OK_THRESHOLD = 100

def iso_to_yyyymm(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m")

def build_cmd(row) -> str:
    start_ym = iso_to_yyyymm(row["start_iso"])
    end_ym   = iso_to_yyyymm(row["end_iso"])

    return (
        f"python {PIPELINE} "
        f"--tile-shp {TILE_SHP} "
        f"--tile-id {row['scene']} "
        f"--span-years {SPAN_YEARS} "
        f"--run-download "
        f"--season-core-start {start_ym} "
        f"--season-core-end {end_ym} "
        f"--sr-only"
    )

# ------------------------------------------------------------
# Cohorts (SR-only)
# ------------------------------------------------------------
df = master_df.copy()
df["sr_tiles"] = df["sr_tiles"].fillna(0)

missing_sr_df = (
    df[df["sr_tiles"] == 0]
    .sort_values(["scene", "start_iso", "end_iso"])
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])
)

insufficient_sr_df = (
    df[(df["sr_tiles"] > 0) & (df["sr_tiles"] < SR_OK_THRESHOLD)]
    .sort_values(["sr_tiles", "scene", "start_iso"])
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])
)

print(f"Missing SR (sr_tiles == 0): {len(missing_sr_df)}")
print(f"Insufficient SR (0 < sr_tiles < {SR_OK_THRESHOLD}): {len(insufficient_sr_df)}\n")

# ------------------------------------------------------------
# Print commands (limit output so notebook stays readable)
# ------------------------------------------------------------
MAX_PRINT = 50

def print_cmds(label: str, subdf):
    print("=" * 90)
    print(label)
    print("=" * 90)
    for _, row in subdf.head(MAX_PRINT).iterrows():
        print(f"# {row['scene']}  ({row['start_iso']} → {row['end_iso']}) | sr_tiles={int(row['sr_tiles'])}")
        print(build_cmd(row))
        print()

print_cmds("MISSING SR (rerun needed)", missing_sr_df)
print_cmds(f"INSUFFICIENT SR (<{SR_OK_THRESHOLD}, rerun likely)", insufficient_sr_df)


Missing SR (sr_tiles == 0): 145
Insufficient SR (0 < sr_tiles < 100): 0

MISSING SR (rerun needed)
# p089r078  (2023-03-06 → 2023-10-24) | sr_tiles=0
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p089r078 --span-years 10 --run-download --season-core-start 202303 --season-core-end 202310 --sr-only

# p089r084  (2023-01-25 → 2023-10-24) | sr_tiles=0
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p089r084 --span-years 10 --run-download --season-core-start 202301 --season-core-end 202310 --sr-only

# p090r088  (2023-01-08 → 2023-10-15) | sr_tiles=0
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p090r088 --span-years 10 --run-download --seaso

In [6]:
from datetime import datetime

PYTHON_EXE = "/env/bin/python"
MASTER_PIPELINE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"

SR_ROOT = "/home/jovyan/scratch/eds/tiles"
FC_ROOT = "/home/jovyan/scratch/eds/tiles"
OUT_ROOT = "/home/jovyan/scratch/eds/compat/files/optb"

def iso_to_yyyymmdd(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m%d")

def scene_to_tile(scene: str) -> str:
    # convert p095r081 → 095_081
    scene = scene.lower().replace("p", "").replace("r", "_")
    return scene

# --------------------------------------------------
# FILTER: SR only (>= 100 tiles)
# --------------------------------------------------

ready_df = master_df[
    master_df["sr_tiles"].fillna(0) >= 100
].sort_values(["run", "scene", "start_iso"])

print(f"Total SR-ready jobs: {len(ready_df)}\n")

# --------------------------------------------------
# GROUP BY RUN
# --------------------------------------------------

for run_name, group in ready_df.groupby("run"):

    print("=" * 80)
    print(f"# RUN: {run_name}")
    print("=" * 80)

    for _, row in group.iterrows():

        start_str = iso_to_yyyymmdd(row["start_iso"])
        end_str   = iso_to_yyyymmdd(row["end_iso"])
        tile_str  = scene_to_tile(row["scene"])

        cmd = (
            f'{PYTHON_EXE} "{MASTER_PIPELINE}" '
            f'--tile {tile_str} '
            f'--start-date {start_str} '
            f'--end-date {end_str} '
            f'--sr-root {SR_ROOT} '
            f'--fc-root {FC_ROOT} '
            f'--out-root "{OUT_ROOT}" '
            f'--timeseries-source ndvi '
            f'--fc-only-clr '
            f'--sr-only-clr '
            f'--python-exe "{PYTHON_EXE}" '
            f'--diagnostics '
            f'--force-compat'
        )

        print(cmd)
    print("\n")


Total SR-ready jobs: 0



# ------------------- CHECK lsat data

In [7]:
import subprocess
from pathlib import Path

# --- Local ---
LOCAL_SCRATCH = Path.home() / "scratch"

# --- S3 ---
S3_SCRATCH = "s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor"
S3_PROJECT = "s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor"

print("Local scratch :", LOCAL_SCRATCH)
print("S3 scratch    :", S3_SCRATCH)
print("S3 project    :", S3_PROJECT)
print("\n")

# -------------------------------------------------------
# Helper to run AWS CLI and capture output
# -------------------------------------------------------

def run_aws(cmd):
    print(">>>", " ".join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("ERROR:\n", result.stderr)
        return None
    return result.stdout

# -------------------------------------------------------
# 1️ Check bucket access
# -------------------------------------------------------

print("Checking bucket access...\n")

run_aws(["aws", "s3", "ls", S3_SCRATCH])
print("-" * 60)
run_aws(["aws", "s3", "ls", S3_PROJECT])
print("\n")

# -------------------------------------------------------
# 2️ Check eds prefix under each
# -------------------------------------------------------

print("Checking eds prefix...\n")

scratch_eds = f"{S3_SCRATCH}/eds/"
project_eds = f"{S3_PROJECT}/eds/"

scratch_out = run_aws(["aws", "s3", "ls", scratch_eds])
print("-" * 60)
project_out = run_aws(["aws", "s3", "ls", project_eds])
print("\n")

# -------------------------------------------------------
# 3️ Count number of files under eds (recursive)
# -------------------------------------------------------

def count_files(s3_path):
    result = subprocess.run(
        ["aws", "s3", "ls", s3_path, "--recursive"],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        print("ERROR:", result.stderr)
        return 0
    
    lines = result.stdout.strip().split("\n")
    if lines == ['']:
        return 0
    return len(lines)

print("File counts under eds/ ...\n")

scratch_count = count_files(scratch_eds)
project_count = count_files(project_eds)

print(f"S3 scratch eds file count : {scratch_count}")
print(f"S3 project eds file count : {project_count}")


Local scratch : /home/jovyan/scratch
S3 scratch    : s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor
S3 project    : s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor


Checking bucket access...

>>> aws s3 ls s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor
------------------------------------------------------------
>>> aws s3 ls s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor


Checking eds prefix...

>>> aws s3 ls s3://dcceew-prod-user-scratch/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/
------------------------------------------------------------
>>> aws s3 ls s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/


File counts under eds/ ...

S3 scratch eds file count : 38875
S3 project eds file count : 4199


In [8]:
def total_size_mb(s3_path):
    result = subprocess.run(
        ["aws", "s3", "ls", s3_path, "--recursive"],
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        return 0
    
    total_bytes = 0
    for line in result.stdout.splitlines():
        parts = line.split()
        if len(parts) >= 4:
            total_bytes += int(parts[2])
    
    return total_bytes / (1024 * 1024)

print(f"S3 scratch size (MB): {total_size_mb(scratch_eds):.2f}")
print(f"S3 project size (MB): {total_size_mb(project_eds):.2f}")


S3 scratch size (MB): 20826204.60
S3 project size (MB): 3953250.41


In [9]:
from datetime import datetime
from pathlib import Path
import subprocess

# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------

PIPELINE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py"
TILE_SHP = "/home/jovyan/assets/eds_lsat_grid_min_max.shp"
SPAN_YEARS = 10
SR_OK_THRESHOLD = 200

LOCAL_TILES = Path.home() / "scratch" / "eds" / "tiles"
S3_PROJECT = "s3://dcceew-epp-data/AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles"

MAX_PRINT = 50

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def iso_to_yyyymm(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m")

def build_cmd(row) -> str:
    start_ym = iso_to_yyyymm(row["start_iso"])
    end_ym   = iso_to_yyyymm(row["end_iso"])

    return (
        f"python {PIPELINE} "
        f"--tile-shp {TILE_SHP} "
        f"--tile-id {row['scene']} "
        f"--span-years {SPAN_YEARS} "
        f"--run-download "
        f"--season-core-start {start_ym} "
        f"--season-core-end {end_ym} "
        f"--sr-only"
    )

def local_exists(scene: str) -> bool:
    return (LOCAL_TILES / scene).exists()

def s3_exists(scene: str) -> bool:
    cmd = ["aws", "s3", "ls", f"{S3_PROJECT}/{scene}/"]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.returncode == 0 and result.stdout.strip() != ""

# ------------------------------------------------------------
# Status Classification
# ------------------------------------------------------------

df = master_df.copy()
df["sr_tiles"] = df["sr_tiles"].fillna(0)

status_rows = []

for _, row in df.iterrows():

    scene = row["scene"]

    has_local = local_exists(scene)
    has_s3 = s3_exists(scene)

    if not has_local and not has_s3:
        storage_status = "MISSING_HOME_AND_PROJECT"
    elif has_local and not has_s3:
        storage_status = "LOCAL_ONLY"
    elif not has_local and has_s3:
        storage_status = "PROJECT_ONLY"
    else:
        storage_status = "BOTH"

    status_rows.append(storage_status)

df["storage_status"] = status_rows

# ------------------------------------------------------------
# Identify rerun groups
# ------------------------------------------------------------

missing_everywhere = df[df["storage_status"] == "MISSING_HOME_AND_PROJECT"]

insufficient_sr = df[
    (df["sr_tiles"] > 0) &
    (df["sr_tiles"] < SR_OK_THRESHOLD)
]

print("\n=== STORAGE SUMMARY ===")
print(df["storage_status"].value_counts())
print()

print(f"Missing everywhere        : {len(missing_everywhere)}")
print(f"Insufficient SR (<{SR_OK_THRESHOLD}) : {len(insufficient_sr)}\n")

# ------------------------------------------------------------
# Print rerun commands
# ------------------------------------------------------------

def print_cmds(label, subdf):
    print("=" * 100)
    print(label)
    print("=" * 100)

    for _, row in subdf.head(MAX_PRINT).iterrows():
        print(f"# {row['scene']} | {row['start_iso']} → {row['end_iso']} "
              f"| sr_tiles={int(row['sr_tiles'])} "
              f"| storage={row['storage_status']}")
        print(build_cmd(row))
        print()

print_cmds("MISSING EVERYWHERE (full rerun required)", missing_everywhere)
print_cmds(f"INSUFFICIENT SR (<{SR_OK_THRESHOLD})", insufficient_sr)



=== STORAGE SUMMARY ===
storage_status
MISSING_HOME_AND_PROJECT    191
PROJECT_ONLY                 26
LOCAL_ONLY                    1
BOTH                          1
Name: count, dtype: int64

Missing everywhere        : 191
Insufficient SR (<200) : 0

MISSING EVERYWHERE (full rerun required)
# p093r086 | 2023-01-14 → 2023-09-11 | sr_tiles=0 | storage=MISSING_HOME_AND_PROJECT
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p093r086 --span-years 10 --run-download --season-core-start 202301 --season-core-end 202309 --sr-only

# p093r087 | 2023-01-14 → 2023-10-29 | sr_tiles=0 | storage=MISSING_HOME_AND_PROJECT
python /home/jovyan/work-easi-eds/scripts/easi-scripts/eds_lsat_collection/ls89_fc_sr_pipeline.py --tile-shp /home/jovyan/assets/eds_lsat_grid_min_max.shp --tile-id p093r087 --span-years 10 --run-download --season-core-start 202301 --season-core-end 202310 --sr-only

In [10]:
# ------------------------------------------------------------
# SINGLE TILE CHECK + PROMPT
# ------------------------------------------------------------

def prompt_for_tile(tile_id: str, run: str | None = None):
    sub = df[df["scene"] == tile_id].copy()

    if run is not None:
        sub = sub[sub["run"] == run]

    if sub.empty:
        print(f"No entry found for tile={tile_id}" + (f", run={run}" if run else ""))
        return

    sub = sub.sort_values(["start_iso", "end_iso"])

    for _, row in sub.iterrows():

        has_local = local_exists(tile_id)
        has_s3 = s3_exists(tile_id)

        if not has_local and not has_s3:
            storage_status = "MISSING_HOME_AND_PROJECT"
        elif has_local and not has_s3:
            storage_status = "LOCAL_ONLY"
        elif not has_local and has_s3:
            storage_status = "PROJECT_ONLY"
        else:
            storage_status = "BOTH"

        sr_tiles = int(row["sr_tiles"])

        print("=" * 100)
        print(f"TILE: {tile_id}")
        print(f"Run: {row.get('run')}")
        print(f"Date range: {row['start_iso']} → {row['end_iso']}")
        print(f"SR tiles: {sr_tiles}")
        print(f"Storage: {storage_status}")
        print("=" * 100)

        if sr_tiles >= SR_OK_THRESHOLD and storage_status == "PROJECT_ONLY":
            print(" Looks complete and archived — no rerun needed.\n")
        else:
            print("  Rerun recommended:\n")
            print(build_cmd(row))
            print()


In [11]:
prompt_for_tile("p094r086", run="run01")


No entry found for tile=p094r086, run=run01


In [13]:
from datetime import datetime
from pathlib import Path

# -----------------------------
# Config (SR-only)
# -----------------------------
PY = "/env/bin/python"
PIPE = "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py"

# raw SR tiles live here (home scratch)
SR_ROOT = "/home/jovyan/scratch/eds/tiles"

# compat output root
OUT = "/home/jovyan/scratch/eds/compat/files/optb"

# timeseries folder under compat/files/
TIMESERIES = "ndvi"

# compat root used for “already exists” checks
COMPAT_ROOT = Path("/home/jovyan/scratch/eds/compat")
COMPAT_FILES_ROOT = COMPAT_ROOT / "files"

# minimum SR tiles required in master_df to be considered “ready”
MIN_SR_TILES = 10

# if True: skip runs where outputs already exist (based on REQUIRE_DATE_WINDOW)
SKIP_IF_EXISTS = False

# skip behaviour:
# - False: treat ANY output for that scene under TIMESERIES as “exists”
# - True : require outputs that match the exact start/end stamp
REQUIRE_DATE_WINDOW = True


# -----------------------------
# Helpers
# -----------------------------
def iso_to_yyyymmdd(iso_date: str) -> str:
    return datetime.strptime(iso_date, "%Y-%m-%d").strftime("%Y%m%d")


def scene_to_tile(scene: str) -> str:
    # p089r078 → 089_078
    scene = str(scene).lower().strip()
    return f"{scene[1:4]}_{scene[5:8]}"


def build_eds_cmd(row) -> str:
    scene = str(row["scene"])
    tile = scene_to_tile(scene)
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    return (
        f'{PY} "{PIPE}" '
        f'--tile {tile} '
        f'--start-date {start} '
        f'--end-date {end} '
        f'--sr-root {SR_ROOT} '
        f'--out-root "{OUT}" '
        f'--timeseries-source {TIMESERIES} '
        f'--sr-only-clr '
        f'--python-exe "{PY}" '
        f'--diagnostics '
        f'--force-compat'
    )


def local_sr_exists(scene: str) -> bool:
    """
    True if SR data exists locally for this scene:
      /home/jovyan/scratch/eds/tiles/<scene>/sr/...
    """
    base = Path(SR_ROOT) / scene / "sr"
    return base.exists() and any(base.rglob("*"))


def compat_outputs_exist(row, require_date_window: bool = True) -> bool:
    """
    Returns True if compat outputs already exist for this scene under:
      /home/jovyan/scratch/eds/compat/files/<TIMESERIES>/<scene>/...

    If require_date_window=True, it also requires finding the start/end stamp.
    """
    scene = str(row["scene"])
    start = iso_to_yyyymmdd(row["start_iso"])
    end = iso_to_yyyymmdd(row["end_iso"])

    ts_root = COMPAT_FILES_ROOT / TIMESERIES
    if not ts_root.exists():
        return False

    scene_dir = ts_root / scene
    if not scene_dir.exists():
        return False

    # any content at all?
    if not any(scene_dir.rglob("*")):
        return False

    if not require_date_window:
        return True

    # date-window-specific patterns (cover common naming)
    stamp_patterns = [
        f"*d{start}_{end}*",
        f"*{start}*{end}*",
    ]
    for pat in stamp_patterns:
        if any(scene_dir.rglob(pat)):
            return True

    return False


# -----------------------------
# Main: filter + print commands (SR-only)
# -----------------------------
df = master_df.copy()

# normalise SR tiles
df["sr_tiles"] = df["sr_tiles"].fillna(0)

# SR-ready based on master_df stats
ready_df = (
    df[df["sr_tiles"] >= MIN_SR_TILES]
    .sort_values(["scene", "start_iso", "end_iso"])
    .drop_duplicates(subset=["scene", "start_iso", "end_iso"])
)

print(
    f"SR-ready tiles (sr_tiles >= {MIN_SR_TILES}) "
    f"for TIMESERIES='{TIMESERIES}': {len(ready_df)}\n"
)

printed = 0
skipped_exists = 0
skipped_no_sr = 0

for _, row in ready_df.iterrows():
    scene = str(row["scene"])

    # must have SR data locally to run compat
    if not local_sr_exists(scene):
        skipped_no_sr += 1
        print(
            f"# SKIP {scene} ({row['start_iso']} → {row['end_iso']}) "
            f"— no local SR data under {Path(SR_ROOT)/scene/'sr'}"
        )
        continue

    exists = compat_outputs_exist(row, require_date_window=REQUIRE_DATE_WINDOW)

    if SKIP_IF_EXISTS and exists:
        skipped_exists += 1
        print(
            f"# SKIP {scene} ({row['start_iso']} → {row['end_iso']}) "
            f"— outputs exist for '{TIMESERIES}'"
        )
        continue

    printed += 1
    note = "REDO (exists)" if exists else "NEW"

    print(
        f"# RUN {scene} ({row['start_iso']} → {row['end_iso']}) "
        f"[{TIMESERIES}] — {note} | sr_tiles={int(row['sr_tiles'])}"
    )
    print(build_eds_cmd(row))
    print()

print(
    "\nSummary:\n"
    f"  printed commands        : {printed}\n"
    f"  skipped (no local SR)   : {skipped_no_sr}\n"
    f"  skipped (already exists): {skipped_exists}\n"
)


SR-ready tiles (sr_tiles >= 10) for TIMESERIES='ndvi': 1

# RUN p093r081 (2023-03-19 → 2023-10-21) [ndvi] — NEW | sr_tiles=234
/env/bin/python "/home/jovyan/work-easi-eds/scripts/easi-scripts/eds-processing/easi_eds_master_processing_pipeline.py" --tile 093_081 --start-date 20230319 --end-date 20231021 --sr-root /home/jovyan/scratch/eds/tiles --out-root "/home/jovyan/scratch/eds/compat/files/optb" --timeseries-source ndvi --sr-only-clr --python-exe "/env/bin/python" --diagnostics --force-compat


Summary:
  printed commands        : 1
  skipped (no local SR)   : 0
  skipped (already exists): 0

